# Phase 5: Plan A (hybrid ML) + Plan C (DistilBERT)

Higher-accuracy experiments with **MLflow** logging (`youtube-toxic-detector` experiment).

## Environment (`uv`)

```bash
uv sync
uv run python -m src.pipeline.plan_a_train
uv run python -m src.pipeline.plan_c_train
uv run mlflow ui --backend-store-uri mlruns
```

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

plan_a = json.loads((ROOT / "reports/phase5/plan_a_report.json").read_text())
plan_c = json.loads((ROOT / "reports/phase5/plan_c_report.json").read_text())
phase4 = json.loads((ROOT / "reports/phase4/phase4_optimization.json").read_text())
print("Phase 4:", phase4["best_model"], phase4["metrics"]["f1_toxic"])
print("Plan A:", plan_a["best_model"], plan_a["metrics"]["f1_toxic"])
print("Plan C:", plan_c["base_model"], plan_c["metrics"]["f1_toxic"])

Phase 4: random_forest_tuned {'metric': 'f1_toxic', 'train': 0.7052, 'test': 0.6627, 'gap_pct': 4.25, 'passes': True}
Plan A: optuna_logistic_plan_a {'metric': 'f1_toxic', 'train': 0.6517, 'test': 0.6057, 'gap_pct': 4.6, 'passes': True}
Plan C: distilbert-base-uncased {'metric': 'f1_toxic', 'train': 0.9315, 'test': 0.7941, 'gap_pct': 13.74, 'passes': False}


## Evaluation (train vs test, 5% gap rule)

In [2]:
import pandas as pd
from IPython.display import display

from src.evaluation.report_display import (
    overall_pass_label,
    print_evaluation_banner,
    report_to_evaluation_df,
)

print_evaluation_banner("Phase 5 — model comparison")

comparison_reports = [
    ("Phase 4 RF", phase4),
    ("Plan A hybrid", plan_a),
    ("Plan C DistilBERT v1", plan_c),
]
eval_frames = []
for label, rep in comparison_reports:
    frame = report_to_evaluation_df(rep, model_name=label, include_all_models=False)
    frame["overall_pass"] = overall_pass_label(rep)
    eval_frames.append(frame)

eval_df = pd.concat(eval_frames, ignore_index=True)
display(eval_df)

summary = [
    {
        "model": label,
        "test_f1": rep["metrics"]["f1_toxic"]["test"],
        "f1_gap_pp": rep["metrics"]["f1_toxic"]["gap_pct"],
        "overall_pass": overall_pass_label(rep),
    }
    for label, rep in comparison_reports
]
print(json.dumps(summary, indent=2))

Phase 5 — model comparison — OVERFITTING EVALUATION
Rule: |train − test| < 5 percentage points (accuracy & F1 toxic)


,model,metric,train,test,gap_pp,pass,overall_pass
0,Phase 4 RF,accuracy,0.7512,0.7200,3.12,PASS,PASS
1,Phase 4 RF,f1_toxic,0.7052,0.6627,4.25,PASS,PASS
2,Plan A hybrid,accuracy,0.6154,0.6550,3.96,PASS,PASS
3,Plan A hybrid,f1_toxic,0.6517,0.6057,4.60,PASS,PASS
4,Plan C DistilBERT v1,accuracy,0.9363,0.7900,14.62,FAIL,FAIL
5,Plan C DistilBERT v1,f1_toxic,0.9315,0.7941,13.74,FAIL,FAIL


[
  {
    "model": "Phase 4 RF",
    "test_f1": 0.6627,
    "f1_gap_pp": 4.25,
    "overall_pass": "PASS"
  },
  {
    "model": "Plan A hybrid",
    "test_f1": 0.6057,
    "f1_gap_pp": 4.6,
    "overall_pass": "PASS"
  },
  {
    "model": "Plan C DistilBERT v1",
    "test_f1": 0.7941,
    "f1_gap_pp": 13.74,
    "overall_pass": "FAIL"
  }
]


## MLflow in this repo

**None** of the older notebooks (`01_baseline_phase*.ipynb`, `phase1`–`phase4`) use MLflow.

Runs are logged under `./mlruns` with experiment **`youtube-toxic-detector`**:
- `plan_a_hybrid`
- `plan_c_distilbert`

## Conclusion

See KEY FINDINGS below.

In [3]:
print("=" * 60)
print("PHASE 5 — KEY FINDINGS")
print("=" * 60)
print(f"Phase 4 (gap OK): acc={phase4['metrics']['accuracy']['test']:.1%} F1={phase4['metrics']['f1_toxic']['test']:.1%}")
print(f"Plan A (+aug+hybrid): acc={plan_a['metrics']['accuracy']['test']:.1%} F1={plan_a['metrics']['f1_toxic']['test']:.1%} gap_pass={plan_a['passes_overfitting']}")
print(f"Plan C (DistilBERT): acc={plan_c['metrics']['accuracy']['test']:.1%} F1={plan_c['metrics']['f1_toxic']['test']:.1%} gap_pass={plan_c['passes_overfitting']}")
print("\nHighest test accuracy: Plan C (~78%). API still uses phase4_best (passes 5% gap).")
print("View runs: uv run mlflow ui --backend-store-uri mlruns")
print("=" * 60)

PHASE 5 — KEY FINDINGS
Phase 4 (gap OK): acc=72.0% F1=66.3%
Plan A (+aug+hybrid): acc=65.5% F1=60.6% gap_pass=True
Plan C (DistilBERT): acc=79.0% F1=79.4% gap_pass=False

Highest test accuracy: Plan C (~78%). API still uses phase4_best (passes 5% gap).
View runs: uv run mlflow ui --backend-store-uri mlruns
